# JupyterLite で学ぶ Excel ファイル操作 入門チュートリアル（openpyxl / XlsxWriter）

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、Python から **Excel ファイル（.xlsx）** を
読み書きする方法を学ぶためのチュートリアルです。レポートや提出物が Excel で求められる場面は多く、
「Python で集計 → Excel で提出」ができると仕事の幅が広がります。

## 対象者
- pandas の基本（DataFrame の作成・表示）を知っている方
- Excel ファイルを Python で自動的に作りたい・読み込みたい方
- 集計結果を書式付きの Excel レポートにまとめたい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. Excel ファイルと 3 つのライブラリの役割分担
2. pandas で Excel を読み書きする（複数シート、読み込みオプション）
3. openpyxl でセル単位に読み書きする（数式を含む）
4. openpyxl で書式を設定する（フォント・塗り・罫線・表示形式・列幅）
5. 既存の Excel ファイルを編集する
6. XlsxWriter で書式付きの Excel とグラフを作る（条件付き書式を含む）
7. JupyterLite で作ったファイルをダウンロード・アップロードする
8. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 作成した Excel ファイルは、左のファイルブラウザに表示されます。ダブルクリックしても中身は見られないので、
  右クリック →「Download」で手元に保存して Excel で開いてください（第 7 章）。

---
## 0. 環境準備（JupyterLite 用）

`openpyxl` と `xlsxwriter` は純 Python のライブラリなので、`piplite` で PyPI からインストールできます。

In [ ]:
# JupyterLite 用のパッケージインストール（ローカルの Jupyter ではスキップされます）
try:
    import piplite
    await piplite.install(["openpyxl", "xlsxwriter", "pandas", "numpy", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import numpy as np
import pandas as pd
import openpyxl
import xlsxwriter

print("pandas バージョン    :", pd.__version__)
print("openpyxl バージョン  :", openpyxl.__version__)
print("XlsxWriter バージョン:", xlsxwriter.__version__)

---
## 1. Excel ファイルと 3 つのライブラリの役割分担

Excel の `.xlsx` ファイルは、実は XML ファイルを ZIP でまとめたものです。Python では次の 3 つの道具を使い分けます。

| ライブラリ | 得意なこと | 苦手なこと |
|---|---|---|
| **pandas**（`read_excel` / `to_excel`） | 表（DataFrame）をシートとして丸ごと読み書きする | セル単位の細かい書式 |
| **openpyxl** | セル単位の読み書き、数式、書式、**既存ファイルの編集** | 大量データの高速書き出し |
| **XlsxWriter** | 書式付きの **新規ファイル作成**、グラフ・条件付き書式の埋め込み | 既存ファイルを開くこと（書き込み専用） |

pandas は内部で openpyxl（読み書き）や XlsxWriter（書き込み）を使っています。
「pandas で表を書き出し、openpyxl / XlsxWriter で仕上げる」のが定番の組み合わせです。

---
## 2. pandas で Excel を読み書きする

### 2.1 DataFrame を Excel に書き出す・読み込む

まず、練習用の売上データ（4 店舗 × 12 か月）を作ります。

In [ ]:
rng = np.random.default_rng(7)

stores = ["名古屋駅前店", "栄店", "豊橋店", "岐阜店"]
months = [f"2025-{m:02d}" for m in range(1, 13)]
rows = []
for store in stores:
    base = rng.integers(800, 1500)
    for month in months:
        rows.append({"店舗": store, "月": month, "売上": int(base + rng.normal(0, 120)), "客数": int(rng.integers(300, 900))})
sales = pd.DataFrame(rows)
print(sales.shape)
sales.head()

In [ ]:
# to_excel で書き出す（index=False で行番号を書かない）
sales.to_excel("sales.xlsx", index=False)
print("sales.xlsx を書き出しました（左のファイルブラウザを確認してください）")

In [ ]:
# read_excel で読み込む
loaded = pd.read_excel("sales.xlsx")
print(loaded.dtypes)
loaded.head()

### 2.2 複数のシートに書き出す

`pd.ExcelWriter` を使うと、1 つのファイルに複数のシートを書き込めます。

In [ ]:
# 店舗ごとの集計表を作る
summary = sales.groupby("店舗", as_index=False).agg(年間売上=("売上", "sum"), 月平均売上=("売上", "mean"), 年間客数=("客数", "sum"))
summary["月平均売上"] = summary["月平均売上"].round(1)
print(summary)

with pd.ExcelWriter("report_pandas.xlsx", engine="openpyxl") as writer:
    sales.to_excel(writer, sheet_name="明細", index=False)
    summary.to_excel(writer, sheet_name="店舗別集計", index=False)
print("report_pandas.xlsx を書き出しました")

In [ ]:
# sheet_name=None ですべてのシートを辞書として読み込む
sheets = pd.read_excel("report_pandas.xlsx", sheet_name=None)
print("シート名:", list(sheets.keys()))
for name, df in sheets.items():
    print(name, df.shape)
sheets["店舗別集計"]

### 2.3 読み込みのオプション

| 引数 | 意味 |
|---|---|
| `sheet_name` | シート名または番号（0 始まり）。`None` で全シート |
| `usecols` | 読み込む列（列名のリスト、または `"A:C"` のような Excel の列記号） |
| `nrows` | 先頭から読む行数 |
| `header` | 見出しの行番号（0 始まり）。見出しがなければ `None` |
| `skiprows` | 先頭から読み飛ばす行数 |
| `index_col` | 行のインデックスにする列 |

In [ ]:
part = pd.read_excel("report_pandas.xlsx", sheet_name="明細", usecols=["店舗", "月", "売上"], nrows=5)
print(part)
print()
print(pd.read_excel("report_pandas.xlsx", sheet_name=1, index_col="店舗"))

### 練習問題 1

1. `sales` から「月」ごとの売上合計（全店舗）を求め、`monthly.xlsx` に `index=False` で書き出してください。
2. `monthly.xlsx` を読み込み、売上合計が最大の月を表示してください。
3. `report_pandas.xlsx` の「明細」シートから、`usecols="A:C"` で最初の 3 列だけを読み込んでください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
monthly = sales.groupby("月", as_index=False)["売上"].sum()
monthly.to_excel("monthly.xlsx", index=False)

# 2
m = pd.read_excel("monthly.xlsx")
print(m.loc[m["売上"].idxmax()])

# 3
print(pd.read_excel("report_pandas.xlsx", sheet_name="明細", usecols="A:C").head())
```

</details>

---
## 3. openpyxl でセル単位に読み書きする

pandas は「表を丸ごと」扱うのに対し、openpyxl は Excel の **ブック（Workbook）→ シート（Worksheet）→ セル（Cell）**
という構造をそのまま操作します。

### 3.1 新しいブックを作ってセルに書き込む

In [ ]:
from openpyxl import Workbook

wb = Workbook()            # 新しいブック（シートが 1 枚ある）
ws = wb.active             # 最初のシート
ws.title = "家計簿"         # シート名を変える

# セル番地で書き込む
ws["A1"] = "費目"
ws["B1"] = "予算"
ws["C1"] = "実績"

# 行・列番号で書き込む（1 始まり）
ws.cell(row=2, column=1, value="食費")
ws.cell(row=2, column=2, value=40000)
ws.cell(row=2, column=3, value=43500)

# append で 1 行まとめて追加
ws.append(["住居費", 60000, 60000])
ws.append(["光熱費", 12000, 13800])
ws.append(["通信費", 8000, 7500])
ws.append(["交際費", 15000, 21000])

wb.save("budget.xlsx")
print("budget.xlsx を保存しました:", ws.max_row, "行 ×", ws.max_column, "列")

### 3.2 ブックを読み込んでセルの値を取り出す

In [ ]:
from openpyxl import load_workbook

wb = load_workbook("budget.xlsx")
print("シート一覧:", wb.sheetnames)
ws = wb["家計簿"]

print("B2 の値:", ws["B2"].value)
print("2 行目 3 列目の値:", ws.cell(row=2, column=3).value)

# 行ごとに値だけを取り出す（values_only=True）
for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
    print(row)

In [ ]:
# 範囲を指定して読む → pandas の DataFrame に変換する
data = list(ws.iter_rows(min_row=2, values_only=True))
budget_df = pd.DataFrame(data, columns=[c.value for c in ws[1]])
budget_df["差額"] = budget_df["実績"] - budget_df["予算"]
budget_df

### 3.3 数式を書き込む

セルに `"=B2*C2"` のような文字列を入れると **数式** になります。ただし openpyxl は数式を **計算しません**
（計算するのは Excel 本体です）。`load_workbook(data_only=True)` で読むと、Excel で一度も保存されていない
数式のセルは `None` になる点に注意してください。

In [ ]:
wb = load_workbook("budget.xlsx")
ws = wb["家計簿"]

ws["D1"] = "差額"
for r in range(2, ws.max_row + 1):
    ws[f"D{r}"] = f"=C{r}-B{r}"          # 実績 − 予算

total_row = ws.max_row + 1
ws[f"A{total_row}"] = "合計"
ws[f"B{total_row}"] = f"=SUM(B2:B{total_row - 1})"
ws[f"C{total_row}"] = f"=SUM(C2:C{total_row - 1})"
ws[f"D{total_row}"] = f"=SUM(D2:D{total_row - 1})"
wb.save("budget_formula.xlsx")

print("D2 に入っている数式:", ws["D2"].value)
print("data_only=True で読むと:", load_workbook("budget_formula.xlsx", data_only=True)["家計簿"]["D2"].value, "（Excel で保存するまで未計算）")

### 練習問題 2

1. 新しいブックを作り、シート名を「成績」にして、見出し（氏名・国語・数学）と 3 人分のデータを書き込み、`scores.xlsx` に保存してください。
2. `scores.xlsx` を読み込み、`iter_rows(values_only=True)` で全行を表示してください。
3. D 列に「合計」の見出しと `=B2+C2` 形式の数式を追加して、`scores_formula.xlsx` に保存してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
wb = Workbook()
ws = wb.active
ws.title = "成績"
ws.append(["氏名", "国語", "数学"])
ws.append(["田中", 80, 75])
ws.append(["鈴木", 92, 88])
ws.append(["佐藤", 67, 95])
wb.save("scores.xlsx")

# 2
ws = load_workbook("scores.xlsx")["成績"]
for row in ws.iter_rows(values_only=True):
    print(row)

# 3
wb = load_workbook("scores.xlsx")
ws = wb["成績"]
ws["D1"] = "合計"
for r in range(2, ws.max_row + 1):
    ws[f"D{r}"] = f"=B{r}+C{r}"
wb.save("scores_formula.xlsx")
```

</details>

---
## 4. openpyxl で書式を設定する

見出しを太字にする、金額に桁区切りを付ける、罫線を引く、といった書式は `openpyxl.styles` で設定します。

| クラス | 用途 |
|---|---|
| `Font` | 太字・色・サイズ |
| `PatternFill` | セルの塗りつぶし |
| `Alignment` | 配置（中央揃えなど） |
| `Border`, `Side` | 罫線 |
| `cell.number_format` | 表示形式（`"#,##0"`, `"0.0%"`, `"yyyy/mm/dd"` など） |

In [ ]:
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

wb = load_workbook("budget_formula.xlsx")
ws = wb["家計簿"]

# 見出し行：太字・白文字・青い背景・中央揃え
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(fill_type="solid", start_color="1F4E79")
for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")

# 数値列：3 桁区切り
for row in ws.iter_rows(min_row=2, min_col=2, max_col=4):
    for cell in row:
        cell.number_format = "#,##0"

# 合計行：太字
for cell in ws[ws.max_row]:
    cell.font = Font(bold=True)

# 罫線：全セルに細い線
thin = Side(style="thin", color="999999")
for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column):
    for cell in row:
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

wb.save("budget_formatted.xlsx")
print("budget_formatted.xlsx を保存しました")

### 4.1 列幅・行の固定・セルの結合

In [ ]:
wb = load_workbook("budget_formatted.xlsx")
ws = wb["家計簿"]

ws.column_dimensions["A"].width = 14      # 列幅（文字数の目安）
for col in ["B", "C", "D"]:
    ws.column_dimensions[col].width = 12

ws.freeze_panes = "A2"                    # 1 行目を固定（スクロールしても見出しが残る）

# 表の上にタイトル行を挿入して結合する
ws.insert_rows(1)
ws["A1"] = "2025 年 1 月 家計簿"
ws.merge_cells("A1:D1")
ws["A1"].font = Font(bold=True, size=14)
ws["A1"].alignment = Alignment(horizontal="center")
ws.freeze_panes = "A3"

wb.save("budget_formatted.xlsx")
print("列幅・固定・結合を設定しました")

### 4.2 日付と割合の表示形式

Python の `date` / `datetime` をそのまま書き込むと Excel の日付になります。`number_format` で見た目を整えます。

In [ ]:
from datetime import date

wb = Workbook()
ws = wb.active
ws.title = "予算達成率"
ws.append(["日付", "費目", "予算", "実績", "達成率"])
for i, (cat, budget, actual) in enumerate([("食費", 40000, 43500), ("住居費", 60000, 60000), ("交際費", 15000, 21000)], start=2):
    ws.cell(row=i, column=1, value=date(2025, i - 1, 1))
    ws.cell(row=i, column=2, value=cat)
    ws.cell(row=i, column=3, value=budget)
    ws.cell(row=i, column=4, value=actual)
    ws.cell(row=i, column=5, value=actual / budget)      # 割合は 0〜1 の数値で入れる
    ws.cell(row=i, column=1).number_format = "yyyy/mm/dd"
    ws.cell(row=i, column=5).number_format = "0.0%"       # 表示だけ % にする
ws.column_dimensions["A"].width = 12
wb.save("budget_rate.xlsx")

for row in ws.iter_rows(min_row=1, values_only=True):
    print(row)

### 練習問題 3

1. `scores_formula.xlsx` を開き、見出し行を太字・薄い黄色（`"FFF2CC"`）の背景にして保存してください。
2. 合計列（D 列）の表示形式を `"0"` に、A 列の幅を 12 にしてください。
3. 1 行目を固定（`freeze_panes`）して `scores_styled.xlsx` として保存してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
wb = load_workbook("scores_formula.xlsx")
ws = wb["成績"]

# 1
for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(fill_type="solid", start_color="FFF2CC")

# 2
for r in range(2, ws.max_row + 1):
    ws[f"D{r}"].number_format = "0"
ws.column_dimensions["A"].width = 12

# 3
ws.freeze_panes = "A2"
wb.save("scores_styled.xlsx")
print("保存しました")
```

</details>

---
## 5. 既存の Excel ファイルを編集する

既存のファイルに **シートを追加する・値を書き換える・シートを削除する** のは openpyxl の得意分野です
（XlsxWriter は新規作成専用で、既存ファイルを開けません）。

In [ ]:
wb = load_workbook("report_pandas.xlsx")   # pandas で作ったファイルを開く
print("編集前のシート:", wb.sheetnames)

# 1. 値を書き換える（明細シートの最初の売上を修正）
ws = wb["明細"]
print("修正前 C2:", ws["C2"].value)
ws["C2"] = ws["C2"].value + 100
print("修正後 C2:", ws["C2"].value)

# 2. シートを追加してメモを書く
memo = wb.create_sheet("メモ")
memo["A1"] = "作成日"
memo["B1"] = "2025-12-31"
memo["A2"] = "備考"
memo["B2"] = "C2 の売上を修正済み"

# 3. 不要なシートを削除する（例として一度追加してから削除）
tmp = wb.create_sheet("一時")
wb.remove(tmp)

wb.save("report_edited.xlsx")
print("編集後のシート:", wb.sheetnames)

In [ ]:
# pandas で読み直して確認
print(pd.read_excel("report_edited.xlsx", sheet_name="明細").head(3))
print(pd.read_excel("report_edited.xlsx", sheet_name="メモ", header=None))

### 5.1 openpyxl でもグラフを作れる

グラフは次章の XlsxWriter が得意ですが、**既存ファイルにグラフを追加したい** ときは openpyxl の `openpyxl.chart` を使います。
`Reference` でデータ範囲を指定し、`ws.add_chart()` で貼り付けます。

In [ ]:
from openpyxl.chart import BarChart, Reference

wb = load_workbook("report_edited.xlsx")
ws = wb["店舗別集計"]

chart = BarChart()
chart.title = "店舗別の年間売上"
chart.y_axis.title = "売上"
values = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)   # B1:B5（見出しを含める）
labels = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)   # A2:A5（店舗名）
chart.add_data(values, titles_from_data=True)
chart.set_categories(labels)
chart.legend = None
ws.add_chart(chart, "F2")

wb.save("report_chart.xlsx")
print("report_chart.xlsx にグラフを追加しました")

---
## 6. XlsxWriter で書式付きの Excel とグラフを作る

XlsxWriter は「書式付きのファイルを最初から作る」ためのライブラリです。**書式（Format）を先に作り、書き込むときに渡す**
のが特徴で、グラフや条件付き書式も埋め込めます。ファイルは `workbook.close()` で保存されます。

### 6.1 基本の書き込みと書式

In [ ]:
workbook = xlsxwriter.Workbook("xw_basic.xlsx")
ws = workbook.add_worksheet("店舗別集計")

# 書式を作る
bold = workbook.add_format({"bold": True, "bg_color": "#DDEBF7", "border": 1})
money = workbook.add_format({"num_format": "#,##0", "border": 1})
plain = workbook.add_format({"border": 1})

# 見出し行（write_row: 1 行をまとめて書く。行・列は 0 始まり）
ws.write_row(0, 0, ["店舗", "年間売上", "年間客数"], bold)

# データ行
for i, row in summary.iterrows():
    ws.write(i + 1, 0, row["店舗"], plain)
    ws.write(i + 1, 1, row["年間売上"], money)
    ws.write(i + 1, 2, row["年間客数"], money)

ws.set_column(0, 0, 16)      # A 列の幅
ws.set_column(1, 2, 12)      # B〜C 列の幅
workbook.close()
print("xw_basic.xlsx を保存しました")

### 6.2 グラフを埋め込む

`add_chart()` でグラフを作り、`add_series()` でデータ範囲を指定し、`insert_chart()` でシートに貼り付けます。
データ範囲は `[シート名, 先頭行, 先頭列, 末尾行, 末尾列]`（0 始まり）で指定します。

In [ ]:
n = len(summary)
workbook = xlsxwriter.Workbook("xw_chart.xlsx")
ws = workbook.add_worksheet("店舗別集計")
bold = workbook.add_format({"bold": True})
money = workbook.add_format({"num_format": "#,##0"})

ws.write_row(0, 0, ["店舗", "年間売上", "年間客数"], bold)
ws.write_column(1, 0, summary["店舗"])
ws.write_column(1, 1, summary["年間売上"], money)
ws.write_column(1, 2, summary["年間客数"], money)
ws.set_column(0, 0, 16)

# 棒グラフ
chart = workbook.add_chart({"type": "column"})
chart.add_series({
    "name": "年間売上",
    "categories": ["店舗別集計", 1, 0, n, 0],   # A2:A5（店舗名）
    "values": ["店舗別集計", 1, 1, n, 1],       # B2:B5（年間売上）
    "fill": {"color": "#4472C4"},
})
chart.set_title({"name": "店舗別の年間売上"})
chart.set_y_axis({"name": "売上", "num_format": "#,##0"})
chart.set_legend({"none": True})
ws.insert_chart("E2", chart)               # E2 の位置に貼り付け

workbook.close()
print("xw_chart.xlsx を保存しました（グラフ付き）")

In [ ]:
# 折れ線グラフ：月別売上（名古屋駅前店）
nagoya = sales[sales["店舗"] == "名古屋駅前店"].reset_index(drop=True)
n = len(nagoya)

workbook = xlsxwriter.Workbook("xw_line.xlsx")
ws = workbook.add_worksheet("月別")
ws.write_row(0, 0, ["月", "売上", "客数"], workbook.add_format({"bold": True}))
ws.write_column(1, 0, nagoya["月"])
ws.write_column(1, 1, nagoya["売上"])
ws.write_column(1, 2, nagoya["客数"])

line = workbook.add_chart({"type": "line"})
line.add_series({"name": "売上", "categories": ["月別", 1, 0, n, 0], "values": ["月別", 1, 1, n, 1], "marker": {"type": "circle"}})
line.add_series({"name": "客数", "categories": ["月別", 1, 0, n, 0], "values": ["月別", 1, 2, n, 2], "y2_axis": True})
line.set_title({"name": "名古屋駅前店の月別売上と客数"})
line.set_y_axis({"name": "売上"})
line.set_y2_axis({"name": "客数"})
ws.insert_chart("E2", line, {"x_scale": 1.5, "y_scale": 1.2})
workbook.close()
print("xw_line.xlsx を保存しました")

### 6.3 条件付き書式

`conditional_format()` で、値に応じてセルの色を変えられます。

| type | 例 |
|---|---|
| `"cell"` | `{"type": "cell", "criteria": ">", "value": 1200, "format": red}` |
| `"3_color_scale"` | 値の大小を 3 色のグラデーションで表す |
| `"data_bar"` | セル内に棒グラフを表示 |

In [ ]:
workbook = xlsxwriter.Workbook("xw_conditional.xlsx")
ws = workbook.add_worksheet("月別")
ws.write_row(0, 0, ["月", "売上", "客数"], workbook.add_format({"bold": True}))
ws.write_column(1, 0, nagoya["月"])
ws.write_column(1, 1, nagoya["売上"])
ws.write_column(1, 2, nagoya["客数"])

red = workbook.add_format({"bg_color": "#FFC7CE", "font_color": "#9C0006"})
# 売上が平均より高いセルを赤く
avg_sales = float(nagoya["売上"].mean())
ws.conditional_format(f"B2:B{n + 1}", {"type": "cell", "criteria": ">", "value": avg_sales, "format": red})
# 客数は 3 色スケール
ws.conditional_format(f"C2:C{n + 1}", {"type": "3_color_scale"})
workbook.close()
print("xw_conditional.xlsx を保存しました（平均 %.0f より大きい売上を強調）" % avg_sales)

### 6.4 pandas と XlsxWriter を組み合わせる

`pd.ExcelWriter(engine="xlsxwriter")` を使うと、pandas で表を書き出したあと、`writer.book`（Workbook）と
`writer.sheets[シート名]`（Worksheet）を取り出して XlsxWriter の機能で仕上げができます。

In [ ]:
with pd.ExcelWriter("pandas_xlsxwriter.xlsx", engine="xlsxwriter") as writer:
    summary.to_excel(writer, sheet_name="店舗別集計", index=False)
    workbook = writer.book
    ws = writer.sheets["店舗別集計"]

    # 列幅と表示形式
    money = workbook.add_format({"num_format": "#,##0"})
    ws.set_column(0, 0, 16)
    ws.set_column(1, 3, 12, money)

    # グラフ
    n = len(summary)
    chart = workbook.add_chart({"type": "bar"})
    chart.add_series({"name": "年間客数", "categories": ["店舗別集計", 1, 0, n, 0], "values": ["店舗別集計", 1, 3, n, 3]})
    chart.set_title({"name": "店舗別の年間客数"})
    ws.insert_chart("F2", chart)

print("pandas_xlsxwriter.xlsx を保存しました")

### 練習問題 4

1. XlsxWriter で `monthly_xw.xlsx` を作り、月ごとの全店舗売上合計（練習問題 1 の `monthly`）を見出し付きで書き込んでください。
2. 同じシートに、月別売上の折れ線グラフを `E2` に貼り付けてください。
3. 売上合計が 5,000 を超える月のセルに条件付き書式（赤い背景）を付けてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
monthly = sales.groupby("月", as_index=False)["売上"].sum()
n = len(monthly)

workbook = xlsxwriter.Workbook("monthly_xw.xlsx")
ws = workbook.add_worksheet("月別")
bold = workbook.add_format({"bold": True})
money = workbook.add_format({"num_format": "#,##0"})

# 1
ws.write_row(0, 0, ["月", "売上合計"], bold)
ws.write_column(1, 0, monthly["月"])
ws.write_column(1, 1, monthly["売上"], money)

# 2
line = workbook.add_chart({"type": "line"})
line.add_series({"name": "売上合計", "categories": ["月別", 1, 0, n, 0], "values": ["月別", 1, 1, n, 1]})
line.set_title({"name": "月別売上合計"})
ws.insert_chart("E2", line)

# 3
red = workbook.add_format({"bg_color": "#FFC7CE"})
ws.conditional_format(f"B2:B{n + 1}", {"type": "cell", "criteria": ">", "value": 5000, "format": red})
workbook.close()
print("monthly_xw.xlsx を保存しました")
```

</details>

---
## 7. JupyterLite で作ったファイルをダウンロード・アップロードする

JupyterLite で書き出したファイルは **ブラウザの中（ローカルストレージ）** に保存されています。
PC に保存して Excel で開くには、次の手順を使います。

1. 左のファイルブラウザで、ファイル名（例: `xw_chart.xlsx`）を **右クリック**
2. **Download** を選ぶ → PC のダウンロードフォルダに保存される
3. Excel で開く（数式や条件付き書式、グラフは Excel が表示・計算する）

逆に、手元の Excel ファイルを読み込みたいときは、ファイルブラウザ上部の **Upload（↑ のアイコン）** から
アップロードすれば、`pd.read_excel("ファイル名.xlsx")` で読み込めます。

> **注意**: ブラウザのデータを消去すると、JupyterLite 内のファイルも消えます。大事なファイルは必ずダウンロードしてください。

In [ ]:
import os

xlsx_files = sorted(f for f in os.listdir(".") if f.endswith(".xlsx"))
print("このフォルダにある Excel ファイル:")
for f in xlsx_files:
    print(f"  {f}  ({os.path.getsize(f):,} バイト)")

---
## 8. まとめ

| やりたいこと | 使うもの | 主な関数・クラス |
|---|---|---|
| 表を書き出す・読み込む | pandas | `to_excel()`, `read_excel()`, `ExcelWriter` |
| 複数シート | pandas | `ExcelWriter` + `sheet_name`、`read_excel(sheet_name=None)` |
| セル単位の読み書き | openpyxl | `Workbook()`, `load_workbook()`, `ws["A1"]`, `ws.cell()`, `ws.append()`, `iter_rows()` |
| 数式 | openpyxl | `ws["D2"] = "=B2*C2"`（計算は Excel が行う） |
| 書式 | openpyxl | `Font`, `PatternFill`, `Alignment`, `Border`, `number_format`, `column_dimensions`, `freeze_panes`, `merge_cells` |
| 既存ファイルの編集 | openpyxl | `load_workbook()` → 変更 → `save()`、`create_sheet()`, `remove()` |
| 書式付き新規作成 | XlsxWriter | `Workbook()`, `add_worksheet()`, `add_format()`, `write_row()`, `set_column()`, `close()` |
| グラフ | XlsxWriter | `add_chart()`, `add_series()`, `insert_chart()` |
| 条件付き書式 | XlsxWriter | `conditional_format()` |
| pandas と組み合わせ | XlsxWriter | `ExcelWriter(engine="xlsxwriter")`, `writer.book`, `writer.sheets` |

## 次のステップ

- `python/pandas/pandas_intermediate_tutorial.ipynb` — 集計・結合など、Excel に書き出す前のデータ加工
- `python/duckdb/duckdb_sql_beginner_tutorial.ipynb` — SQL で集計した結果を Excel レポートにする
- `python/itables/itables_beginner_tutorial.ipynb` — Excel に出す前にノート上で表を対話的に確認する

---
## 総合演習：家計簿データから Excel レポートを作る

次の手順で、1 年分の家計簿データ（架空）から提出用の Excel レポート `household_report.xlsx` を作ってください。

1. 12 か月 × 5 費目（食費・住居費・光熱費・通信費・交際費）の支出データを乱数で作り、DataFrame `expenses`（列: 月, 費目, 金額）にする。
2. 月ごとの支出合計 `by_month` と、費目ごとの年間合計 `by_category` を pandas で求める。
3. `pd.ExcelWriter(engine="xlsxwriter")` で「明細」「月別集計」「費目別集計」の 3 シートに書き出す。
4. 「月別集計」シートに、金額の桁区切り・列幅を設定し、月別支出の折れ線グラフを貼り付ける。
5. 「月別集計」の金額が年間平均を超える月に条件付き書式（赤）を付ける。
6. 書き出したファイルを pandas で読み直し、3 シートの形（shape）を表示して確認する。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

In [ ]:
rng = np.random.default_rng(2025)
categories = {"食費": 45000, "住居費": 65000, "光熱費": 12000, "通信費": 8000, "交際費": 18000}
months = [f"2025-{m:02d}" for m in range(1, 13)]

# 1. データ作成
rows = []
for month in months:
    for cat, base in categories.items():
        rows.append({"月": month, "費目": cat, "金額": int(base * rng.uniform(0.85, 1.25))})
expenses = pd.DataFrame(rows)

# 2. 集計
by_month = expenses.groupby("月", as_index=False)["金額"].sum()
by_category = expenses.groupby("費目", as_index=False)["金額"].sum().sort_values("金額", ascending=False)
avg_month = float(by_month["金額"].mean())

# 3〜5. Excel に書き出して仕上げる
n = len(by_month)
with pd.ExcelWriter("household_report.xlsx", engine="xlsxwriter") as writer:
    expenses.to_excel(writer, sheet_name="明細", index=False)
    by_month.to_excel(writer, sheet_name="月別集計", index=False)
    by_category.to_excel(writer, sheet_name="費目別集計", index=False)

    workbook = writer.book
    money = workbook.add_format({"num_format": "#,##0"})
    for name in ["明細", "月別集計", "費目別集計"]:
        writer.sheets[name].set_column(0, 1, 12)
    ws = writer.sheets["月別集計"]
    ws.set_column(1, 1, 12, money)

    line = workbook.add_chart({"type": "line"})
    line.add_series({"name": "支出合計", "categories": ["月別集計", 1, 0, n, 0], "values": ["月別集計", 1, 1, n, 1], "marker": {"type": "circle"}})
    line.set_title({"name": "月別支出合計"})
    line.set_y_axis({"num_format": "#,##0"})
    ws.insert_chart("D2", line)

    red = workbook.add_format({"bg_color": "#FFC7CE", "font_color": "#9C0006"})
    ws.conditional_format(f"B2:B{n + 1}", {"type": "cell", "criteria": ">", "value": avg_month, "format": red})

# 6. 読み直して確認
check = pd.read_excel("household_report.xlsx", sheet_name=None)
for name, df in check.items():
    print(name, df.shape)
print("年間平均支出: %.0f 円" % avg_month)
check["費目別集計"]

お疲れさまでした！ 「pandas で表を作り、openpyxl でセルと書式を整え、XlsxWriter でグラフ付きレポートに仕上げる」
という流れが身についていれば、Excel での提出物を Python で自動化できます。